# Empirical Methods HW 7 Cohort 1 Group 5

Fit an ARMA(1,1) to the CMA (investment) Fama-French factor, estimate ARCH(12) and
GARCH(1,1) conditional-variance models on the ARMA residuals by direct MLE, check whether they've
removed volatility clustering, then compare their variance forecasts against realized variance
(RV) built from daily squared returns.

**Data files needed (update the paths below to match your machine):**
- A monthly Fama-French 5-factor CSV (Bloomberg/Ken French style) with columns
  `Date, Mkt-RF, SMB, HML, RMW, CMA, RF`, values in percent, a handful of header rows to skip.
- A **daily** Fama-French (Developed ex US) 5-factor CSV with the same column layout, used to
  build realized variance for CMA.


In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from scipy.optimize import minimize
import matplotlib.pyplot as plt

# Update this path to your local monthly factor file
path = "Hw7FData.csv"

raw = pd.read_csv(path)
raw.head(15)


Bloomberg-style factor downloads carry several metadata rows above the real header, so the
first `read_csv` above is just to inspect the file and find how many rows to skip. Re-load with
`skiprows` set to the row where the real `Date, Mkt-RF, SMB, HML, RMW, CMA, RF` header lives.

In [ ]:
df = pd.read_csv(path, skiprows=6)
df.rename(columns={df.columns[0]: "Date"}, inplace=True)
df.columns = df.columns.str.strip()

print(df.columns)
print(df.head())


In [ ]:
# Keep only real monthly rows (YYYYMM), parse dates, convert percent -> decimal
df["Date"] = df["Date"].astype(str).str.strip()
df = df[df["Date"].str.fullmatch(r"\d{6}")].copy()
df["Date"] = pd.to_datetime(df["Date"], format="%Y%m")
df = df.set_index("Date")

df = df.apply(pd.to_numeric, errors="coerce") / 100.0
df.columns = df.columns.str.strip()

cma = df["CMA"].dropna()
print(cma.index.min(), cma.index.max())
print(cma.head())


## 1) ARMA(1,1) on the CMA factor

In [ ]:
arma = ARIMA(cma, order=(1, 0, 1), trend="c").fit()
print(arma.summary())


In [ ]:
phi = arma.params["ar.L1"]
print("AR(1) persistence (phi):", phi)

half_life = np.log(0.5) / np.log(abs(phi))
print("Half-life (months):", half_life)


## 2) ARCH(12) and GARCH(1,1) on the ARMA residuals

`statsmodels.tsa.api.arch_model` wasn't available in this environment (import error), so both
conditional-variance models are estimated directly by maximizing the Gaussian log-likelihood with
`scipy.optimize.minimize`. Parameters are reparameterized through `exp(.)` / softmax-style ratios
so that `omega > 0`, all ARCH/GARCH weights are non-negative, and for GARCH `alpha + beta < 1`
(and for ARCH(12), `sum(alpha_i) < 1`) automatically — this keeps the optimizer inside the
stationary region without needing explicit inequality constraints.

In [ ]:
eps_series = arma.resid.dropna()
idx = eps_series.index
eps = np.asarray(eps_series)
T = len(eps)


def garch11_negloglik(params, eps):
    # unconstrained
    a0, a1, a2 = params
    # enforce omega > 0
    omega = np.exp(a0)
    # map (a1, a2) -> alpha, beta >= 0 and alpha + beta < 1
    u = np.exp(a1)
    v = np.exp(a2)
    denom = 1 + u + v
    alpha = u / denom
    beta = v / denom

    T = len(eps)
    sigma2 = np.empty(T)

    # initialize at unconditional variance
    sigma2[0] = omega / (1 - alpha - beta)

    for t in range(1, T):
        sigma2[t] = omega + alpha * eps[t - 1] ** 2 + beta * sigma2[t - 1]
        if sigma2[t] <= 0 or not np.isfinite(sigma2[t]):
            return 1e12

    # log-likelihood
    ll = -0.5 * np.sum(np.log(2 * np.pi) + np.log(sigma2) + (eps ** 2) / sigma2)
    return -ll  # minimize negative log-likelihood


init = np.array([np.log(np.var(eps) * 0.85), np.log(0.05), np.log(0.90)])  # starting guess
res_g = minimize(garch11_negloglik, init, args=(eps,), method="Nelder-Mead")

a0, a1, a2 = res_g.x
omega = np.exp(a0)
u, v = np.exp(a1), np.exp(a2)
denom = 1 + u + v
alpha = u / denom
beta = v / denom

print("GARCH(1,1) estimates:")
print("omega =", omega)
print("alpha =", alpha)
print("beta =", beta)
print("alpha+beta =", alpha + beta)
print("Converged:", res_g.success, "|", res_g.message)


In [ ]:
sigma2_g = np.empty(T)
sigma2_g[0] = omega / (1 - alpha - beta)
for t in range(1, T):
    sigma2_g[t] = omega + alpha * eps[t - 1] ** 2 + beta * sigma2_g[t - 1]

sigma2_garch_series = pd.Series(sigma2_g, index=idx)


def arch12_negloglik(params, eps, p=12):
    # params: [a0, a1...ap] unconstrained
    a0 = params[0]
    a = params[1:]

    omega = np.exp(a0)

    u = np.exp(a)                # positive
    denom = 1 + np.sum(u)
    alphas = u / denom           # sum(alphas) < 1 automatically

    T = len(eps)
    sigma2 = np.empty(T)
    eps2 = eps ** 2

    # initialize first p values with unconditional variance
    sigma2[:p] = omega / (1 - np.sum(alphas))

    for t in range(p, T):
        sigma2[t] = omega + np.dot(alphas, eps2[t - p:t][::-1])
        if sigma2[t] <= 0 or not np.isfinite(sigma2[t]):
            return 1e12

    ll = -0.5 * np.sum(np.log(2 * np.pi) + np.log(sigma2[p:]) + eps2[p:] / sigma2[p:])
    return -ll


p = 12
init = np.concatenate(([np.log(np.var(eps) * 0.85)], np.log(np.full(p, 0.02))))
res_a = minimize(arch12_negloglik, init, args=(eps, p), method="Nelder-Mead")

a0 = res_a.x[0]
a = res_a.x[1:]
omega_a = np.exp(a0)
u = np.exp(a)
alphas_a = u / (1 + np.sum(u))

print("ARCH(12) estimates:")
print("omega =", omega_a)
print("sum(alpha_i) =", np.sum(alphas_a))
print("alphas =", alphas_a)
print("Converged:", res_a.success, "|", res_a.message)

sigma2_a = np.empty(T)
eps2 = eps ** 2
sigma2_a[:p] = omega_a / (1 - np.sum(alphas_a))
for t in range(p, T):
    sigma2_a[t] = omega_a + np.dot(alphas_a, eps2[t - p:t][::-1])


**On stationarity and the variance plot:** for the GARCH(1,1) model, `alpha + beta < 1`,
indicating covariance stationarity — that also guarantees a finite unconditional variance. The
ARCH(12) process satisfies the analogous stationarity condition (sum of ARCH coefficients < 1).

Both models capture volatility clustering, with large spikes during crisis periods (dot-com
bubble, global financial crisis, COVID-19). GARCH(1,1) produces smoother, more persistent
variance dynamics than ARCH(12); ARCH is slightly more reactive since its memory is limited to 12
lags, so its variance decays faster after shocks.

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(idx, sigma2_a, label="ARCH(12)")
plt.plot(idx, sigma2_g, label="GARCH(1,1)", alpha=0.8)
plt.legend()
plt.title("Conditional variance from ARCH(12) and GARCH(1,1) on ARMA residuals")
plt.tight_layout()
plt.show()


In [ ]:
# conditional sigmas
sigma_a = np.sqrt(sigma2_a)
sigma_g = np.sqrt(sigma2_g)

# normalized residuals
eta_a = eps / sigma_a
eta_g = eps / sigma_g

abs_eta_a = np.abs(eta_a)
abs_eta_g = np.abs(eta_g)

plt.figure(figsize=(11, 3))
plt.plot(idx, abs_eta_a)
plt.title("|\u03b7_t| from ARCH(12)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 3))
plt.plot(idx, abs_eta_g)
plt.title("|\u03b7_t| from GARCH(1,1)")
plt.tight_layout()
plt.show()


## 3) Did the variance models remove volatility clustering?

The autocorrelation of `|eta_t|` is substantially reduced relative to `eps_t^2`, suggesting that
both ARCH(12) and GARCH(1,1) adequately account for volatility clustering — spikes do not appear
to be too clustered anymore. Lag 0 = 1 by definition; the remaining lags are close to 0, so there
is essentially no persistence left in the magnitude of the standardized shocks.

In [ ]:
from statsmodels.tsa.stattools import acf

max_lag = 36  # monthly data: 36 lags = 3 years

acf_a = acf(abs_eta_a, nlags=max_lag, fft=True)
acf_g = acf(abs_eta_g, nlags=max_lag, fft=True)

plt.figure(figsize=(8, 3))
plt.stem(range(len(acf_a)), acf_a, basefmt=" ")
plt.title("ACF of |\u03b7_t| (ARCH(12))")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 3))
plt.stem(range(len(acf_g)), acf_g, basefmt=" ")
plt.title("ACF of |\u03b7_t| (GARCH(1,1))")
plt.tight_layout()
plt.show()


## 4) Realized variance from daily returns

In [ ]:
# Update this path to your local daily factor file
path_daily = "Developed_ex_US_5_factors_Daily.csv"

d = pd.read_csv(path_daily, skiprows=6)
d.rename(columns={d.columns[0]: "Date"}, inplace=True)
d.columns = d.columns.str.strip()

# keep only daily YYYYMMDD rows
d["Date"] = d["Date"].astype(str).str.strip()
d = d[d["Date"].str.fullmatch(r"\d{8}")].copy()
d["Date"] = pd.to_datetime(d["Date"], format="%Y%m%d")
d = d.set_index("Date")

# percent -> decimal
d = d.apply(pd.to_numeric, errors="coerce") / 100.0
d.columns = d.columns.str.strip()

cma_d = d["CMA"].dropna()

# realized variance per month: sum of squared daily returns in that month
RV = (cma_d ** 2).resample("M").sum()

plt.figure(figsize=(11, 4))
plt.plot(RV.index, RV.values)
plt.title("Monthly Realized Variance of CMA (sum of daily squared returns)")
plt.tight_layout()
plt.show()


Since RV aggregates many daily squared returns within a month, it is a much less noisy proxy for
latent variance than `eps^2`, which is only a single squared innovation — so RV is a much better
estimator of true variance to benchmark the two conditional-variance models against.

In [ ]:
# Align RV, ARMA residuals, and the GARCH conditional variance on a common monthly index
RV_fixed = RV.copy()
RV_fixed.index = pd.to_datetime(RV_fixed.index).to_period("M").to_timestamp("M")

eps_fixed = eps_series.copy()
eps_fixed.index = pd.to_datetime(eps_fixed.index).to_period("M").to_timestamp("M")

sigma_fixed = sigma2_garch_series.copy()
sigma_fixed.index = pd.to_datetime(sigma_fixed.index).to_period("M").to_timestamp("M")

dfq = pd.concat(
    [RV_fixed.rename("RV"), eps_fixed.rename("eps"), sigma_fixed.rename("sigma2_garch")],
    axis=1,
).dropna()

print("Rows after alignment:", len(dfq))
print(dfq.head())


## 5) ARMA(1,1) forecast of realized variance vs. GARCH conditional variance

In [ ]:
rv = dfq["RV"]
arma_rv = ARIMA(rv, order=(1, 0, 1), trend="c").fit()
print(arma_rv.summary())

# v_t = E_{t-1}[RV_t], one-step-ahead forecast (not the in-sample fitted value)
v = arma_rv.get_prediction(start=rv.index[1], dynamic=False).predicted_mean
v = v.rename("v")

# align v with RV and sigma2_garch
df6 = pd.concat([RV.rename("RV"), v, dfq["sigma2_garch"].rename("sigma2_garch")], axis=1).dropna()

corr_v_RV = df6["v"].corr(df6["RV"])
corr_sigma2_RV = df6["sigma2_garch"].corr(df6["RV"])

print("Corr(v_t, RV_t):", corr_v_RV)
print("Corr(sigma^2_GARCH_t, RV_t):", corr_sigma2_RV)

plt.figure(figsize=(11, 4))
plt.plot(df6.index, df6["v"], label="v_t = ARMA(1,1) forecast of RV")
plt.plot(df6.index, df6["sigma2_garch"], label="sigma_t^2 (GARCH)", alpha=0.8)
plt.title("Variance forecasts: ARMA-on-RV vs GARCH conditional variance")
plt.legend()
plt.tight_layout()
plt.show()


**Conclusion:** the correlation between `v_t` (the ARMA(1,1)-on-RV one-step-ahead forecast) and
`RV_t` is higher than the correlation between `sigma^2_t` (GARCH) and `RV_t`. This indicates that
the ARMA(1,1) model applied to realized variance provides a better forecast of future variance
than the GARCH model, which is based only on squared ARMA residuals rather than the richer
intra-month information embedded in realized variance.